In [1]:
import duckdb; 
print(duckdb.__version__)

1.5.4


In [5]:
con = duckdb.connect("synpuf.duckdb")

In [2]:
import os
[f for f in os.listdir() if f.endswith(".csv")]

['DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv',
 'DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv',
 'DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv',
 'DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv']

In [6]:
con.execute("""
    CREATE OR REPLACE TABLE bene_2008 AS
    SELECT * FROM read_csv_auto('DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv')
""")

In [9]:
con.execute("""
    CREATE OR REPLACE TABLE bene_2009 AS
    SELECT * FROM read_csv_auto('DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv')
""")

In [10]:
con.execute("""
    CREATE OR REPLACE TABLE ip_claims AS
    SELECT * FROM read_csv_auto('DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv', all_varchar = true)
""")

In [11]:
con.execute("""
    CREATE OR REPLACE TABLE op_claims AS
    SELECT * FROM read_csv_auto('DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv',
all_varchar = true)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [12]:
con.execute("SHOW TABLES").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,name
0,bene_2008
1,bene_2009
2,ip_claims
3,op_claims


In [13]:
# row counts
for t in ["bene_2008", "bene_2009", "ip_claims", "op_claims"]:
    print(t, con.execute(f"SELECT COUNT(*) FROM {t}").fetchone())

bene_2008 (116352,)
bene_2009 (114538,)
ip_claims (66773,)
op_claims (790790,)


In [ ]:
# row counts
for t in ["bene_2008", "bene_2009", "ip_claims", "op_claims"]:
    print(t, con.execute(f"SELECT COUNT(*) FROM {t}").fetchone())

In [17]:
con.execute("SELECT * FROM ip_claims LIMIT 5").df()

,DESYNPUF_ID,CLM_ID,SEGMENT,CLM_FROM_DT,CLM_THRU_DT,PRVDR_NUM,CLM_PMT_AMT,NCH_PRMRY_PYR_CLM_PD_AMT,AT_PHYSN_NPI,OP_PHYSN_NPI,...,HCPCS_CD_36,HCPCS_CD_37,HCPCS_CD_38,HCPCS_CD_39,HCPCS_CD_40,HCPCS_CD_41,HCPCS_CD_42,HCPCS_CD_43,HCPCS_CD_44,HCPCS_CD_45
0,00013D2EFD8E45D1,196661176988405,1,20100312,20100313,2600GD,4000.00,0.00,3139083564,None,...,None,None,None,None,None,None,None,None,None,None
1,00016F745862898F,196201177000368,1,20090412,20090418,3900MB,26000.00,0.00,6476809087,None,...,None,None,None,None,None,None,None,None,None,None
2,00016F745862898F,196661177015632,1,20090831,20090902,3900HM,5000.00,0.00,0611998537,0611998537,...,None,None,None,None,None,None,None,None,None,None
3,00016F745862898F,196091176981058,1,20090917,20090920,3913XU,5000.00,0.00,4971602784,None,...,None,None,None,None,None,None,None,None,None,None
4,00016F745862898F,196261176983265,1,20100626,20100701,3900MB,16000.00,0.00,6408400473,1960859579,...,None,None,None,None,None,None,None,None,None,None


In [20]:
con.execute("SELECT * FROM bene_2008 LIMIT 5").df()

,DESYNPUF_ID,BENE_BIRTH_DT,BENE_DEATH_DT,BENE_SEX_IDENT_CD,BENE_RACE_CD,BENE_ESRD_IND,SP_STATE_CODE,BENE_COUNTY_CD,BENE_HI_CVRAGE_TOT_MONS,BENE_SMI_CVRAGE_TOT_MONS,...,SP_STRKETIA,MEDREIMB_IP,BENRES_IP,PPPYMT_IP,MEDREIMB_OP,BENRES_OP,PPPYMT_OP,MEDREIMB_CAR,BENRES_CAR,PPPYMT_CAR
0,00013D2EFD8E45D1,19230501,<NA>,1,1,0,26,950,12,12,...,2,0.0,0.0,0.0,50.0,10.0,0.0,0.0,0.0,0.0
1,00016F745862898F,19430101,<NA>,1,1,0,39,230,12,12,...,2,0.0,0.0,0.0,0.0,0.0,0.0,700.0,240.0,0.0
2,0001FDD721E223DC,19360901,<NA>,2,1,0,39,280,12,12,...,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,00021CA6FF03E670,19410601,<NA>,1,5,0,06,290,0,0,...,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,00024B3D2352D2D0,19360801,<NA>,1,1,0,52,590,12,12,...,2,0.0,0.0,0.0,30.0,40.0,0.0,220.0,80.0,0.0


In [14]:
# Unique members per year
for t in ["bene_2008", "bene_2009"]:
    print(t, con.execute(f"SELECT COUNT(DISTINCT DESYNPUF_ID) FROM {t}").fetchone())

bene_2008 (116352,)
bene_2009 (114538,)


In [15]:
# Diabetes and depression prevalence
for t in ["bene_2008", "bene_2009"]:
    print(t, con.execute(f"""
        SELECT
            ROUND(AVG(CASE WHEN SP_DIABETES = 1 THEN 1 ELSE 0 END) * 100, 1) AS pct_diabetes,
            ROUND(AVG(CASE WHEN SP_DEPRESSN = 1 THEN 1 ELSE 0 END) * 100, 1) AS pct_depression
        FROM {t}
    """).fetchone())

bene_2008 (37.9, 21.3)
bene_2009 (41.7, 24.5)


In [16]:
# Rows vs distinct claims
for t in ["ip_claims", "op_claims"]:
    print(t, con.execute(f"SELECT COUNT(*), COUNT(DISTINCT CLM_ID) FROM {t}").fetchone())


ip_claims (66773, 66705)
op_claims (790790, 779815)


In [18]:
# Date range of inpatient claims
print(con.execute("SELECT MIN(CLM_FROM_DT), MAX(CLM_FROM_DT) FROM ip_claims").fetchone())



('20071127', '20101230')


In [19]:
# Inspect one split claim
con.execute("""
    SELECT DESYNPUF_ID, CLM_ID, SEGMENT, CLM_FROM_DT, CLM_THRU_DT, CLM_PMT_AMT
    FROM op_claims
    WHERE CLM_ID = (
        SELECT CLM_ID FROM op_claims
        GROUP BY CLM_ID
        HAVING COUNT(*) > 1
        LIMIT 1
    )
    ORDER BY SEGMENT
""").df()

,DESYNPUF_ID,CLM_ID,SEGMENT,CLM_FROM_DT,CLM_THRU_DT,CLM_PMT_AMT
0,E8E219485DCE5EF3,542992281071207,1,20080218,20080309,2500.00
1,E8E219485DCE5EF3,542992281071207,2,None,None,1700.00


# 2. Build the members table

In [21]:
con.execute("""
CREATE OR REPLACE TABLE members AS
WITH stacked AS (
    SELECT 2008 AS yr, * FROM bene_2008
    UNION ALL BY NAME
    SELECT 2009 AS yr, * FROM bene_2009
)
SELECT
    yr,
    DESYNPUF_ID,
    SP_STATE_CODE                                   AS state,
    BENE_SEX_IDENT_CD                               AS sex,
    yr - CAST(LEFT(CAST(BENE_BIRTH_DT AS VARCHAR), 4) AS INT) AS age,
    CASE WHEN SP_DIABETES = 1 THEN 1 ELSE 0 END     AS diabetes,
    CASE WHEN SP_DEPRESSN =1 THEN 1 ELSE 0 END                AS depression,
    LEAST(BENE_HI_CVRAGE_TOT_MONS, BENE_SMI_CVRAGE_TOT_MONS)             AS member_months,
    ( (SP_ALZHDMTA=1)::INT + (SP_CHF=1)::INT + (SP_CHRNKIDN=1)::INT
    + (SP_CNCR=1)::INT + (SP_COPD=1)::INT + (SP_DEPRESSN=1)::INT
    + (SP_DIABETES=1)::INT + (SP_ISCHMCHT=1)::INT + (SP_OSTEOPRS=1)::INT
    + (SP_RA_OA=1)::INT + (SP_STRKETIA=1)::INT )    AS chronic_count,
    MEDREIMB_IP + MEDREIMB_OP + MEDREIMB_CAR                       AS total_paid
FROM stacked
WHERE BENE_HMO_CVRAGE_TOT_MONS = 0
  AND LEAST(BENE_HI_CVRAGE_TOT_MONS, BENE_SMI_CVRAGE_TOT_MONS) > 0
""")

In [22]:
con.execute("SELECT COUNT(*) FROM members").fetchone()

(152900,)

In [23]:
con.execute("""
    SELECT
        SUM(CASE WHEN BENE_HMO_CVRAGE_TOT_MONS > 0 THEN 1 ELSE 0 END) AS hmo_rows,
        SUM(CASE WHEN BENE_HMO_CVRAGE_TOT_MONS = 0
                  AND LEAST(BENE_HI_CVRAGE_TOT_MONS, BENE_SMI_CVRAGE_TOT_MONS) = 0
                 THEN 1 ELSE 0 END) AS zero_month_rows
    FROM (SELECT * FROM bene_2008 UNION ALL BY NAME SELECT * FROM bene_2009)
""").df()

,hmo_rows,zero_month_rows
0,60220.0,17770.0


In [24]:
con.execute("""
    SELECT * FROM members""").df()

,yr,DESYNPUF_ID,state,sex,age,diabetes,depression,member_months,chronic_count,total_paid
0,2008,00016F745862898F,39,1,65,0,0,12,0,700.0
1,2008,0001FDD721E223DC,39,2,72,0,0,12,0,0.0
2,2008,00024B3D2352D2D0,52,1,72,0,0,12,1,250.0
3,2008,0002F28CE057345B,39,1,86,1,1,12,6,4340.0
4,2008,000308435E3E5B76,24,1,73,0,0,10,1,1020.0
...,...,...,...,...,...,...,...,...,...,...
152895,2009,FFFC880E13CA6278,36,2,73,0,0,12,0,430.0
152896,2009,FFFD1F9EE7AF231B,39,1,70,0,0,12,1,1400.0
152897,2009,FFFD7C464C94D136,54,2,66,0,0,12,0,200.0
152898,2009,FFFE3ED9582AD46E,06,2,93,1,1,12,6,7750.0


In [25]:
con.execute("""
SELECT yr, diabetes,
       COUNT(*)                     AS members,
       SUM(member_months)           AS member_months,
       ROUND(SUM(total_paid) / SUM(member_months) , 0)        AS pmpm
FROM members
GROUP BY yr, diabetes
ORDER BY yr, diabetes
""").df()

,yr,diabetes,members,member_months,pmpm
0,2008,0,46273,527428.0,142.0
1,2008,1,33015,390848.0,696.0
2,2009,0,40545,482573.0,170.0
3,2009,1,33067,396073.0,655.0
